# Laptop ABSA — data exploration

Walks the corpus the training package loads: class balance, aspect
distribution, multi-aspect sentences, mixed-polarity sentences, and what
the encoder actually feeds the model.

Run from `training/notebooks/`.

In [ ]:
import sys
from pathlib import Path

TRAINING_ROOT = Path.cwd().parent
sys.path.insert(0, str(TRAINING_ROOT))

import pandas as pd

from src.data.dataset import (
    LabelMap, label_distribution, load_official_test, load_raw_csv, split_train_val,
)

pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 80)
label_map = LabelMap()
label_map.label2id

## Load

`load_raw_csv` drops `conflict`, realigns the 7 column-shifted rows and
validates `text[from:to] == aspect` on every row. Watch the log line.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s', force=True)

RAW = TRAINING_ROOT / 'data' / 'raw'
train_full = load_raw_csv(RAW / 'Laptop_Train_v2.csv')
test = load_official_test(RAW / 'Laptops_Test_Gold.csv')

print(f'train {len(train_full)} rows, test {len(test)} rows')
train_full.head()

## Class balance

`positive` dominates and `neutral` is the minority — which is why the
trainer weights the loss by inverse frequency and selects on macro F1.

In [ ]:
balance = pd.DataFrame({
    'train': label_distribution(train_full),
    'test': label_distribution(test),
})
balance['train %'] = (100 * balance['train'] / balance['train'].sum()).round(1)
balance['test %'] = (100 * balance['test'] / balance['test'].sum()).round(1)
balance

## Aspects per sentence

The reason the split is grouped by sentence: many sentences carry several
aspects, and some carry *opposing* sentiments across them.

In [ ]:
per_sentence = train_full.groupby('text').size()
print(f'{train_full["text"].nunique()} unique sentences for {len(train_full)} rows')
print(f'mean {per_sentence.mean():.2f} aspects/sentence, max {per_sentence.max()}')
per_sentence.value_counts().sort_index().rename('sentences').to_frame().T

In [ ]:
# Sentences whose aspects disagree -- the examples a sentence-level model cannot get right.
mixed = train_full.groupby('text')['label'].nunique()
mixed_texts = mixed[mixed > 1].index
print(f'{len(mixed_texts)} sentences carry more than one polarity '
      f'({100 * len(mixed_texts) / train_full["text"].nunique():.1f}% of sentences)')

train_full[train_full['text'].isin(mixed_texts[:3])][['text', 'aspect', 'label']]

## Most frequent aspects

In [ ]:
train_full['aspect'].str.lower().value_counts().head(20).rename('count').to_frame()

In [ ]:
# Polarity profile of the most common aspects.
top = train_full['aspect'].str.lower().value_counts().head(10).index
profile = (train_full.assign(aspect_lc=train_full['aspect'].str.lower())
           .query('aspect_lc in @top')
           .pivot_table(index='aspect_lc', columns='label', aggfunc='size', fill_value=0))
profile.loc[top]

## Sentence and aspect length

In [ ]:
lengths = pd.DataFrame({
    'sentence_chars': train_full['text'].str.len(),
    'sentence_words': train_full['text'].str.split().str.len(),
    'aspect_words': train_full['aspect'].str.split().str.len(),
})
lengths.describe(percentiles=[0.5, 0.9, 0.99]).round(1)

## The split

Grouped by sentence, so no sentence appears on both sides.

In [ ]:
train_df, val_df = split_train_val(train_full, val_fraction=0.1, seed=42)

leak = set(train_df['text']) & set(val_df['text'])
print(f'overlapping sentences: {len(leak)}')
print(f'val is {100 * len(val_df) / len(train_full):.1f}% of rows')

pd.DataFrame({'train': label_distribution(train_df), 'val': label_distribution(val_df)})

## What the model actually sees

The encoder builds `[CLS] text [SEP] aspect [SEP]` plus two disjoint masks.
Needs `bert-base-uncased` available locally or via the hub.

In [ ]:
from src.preprocessing.tokenizer import ABSAEncoder

encoder = ABSAEncoder('bert-base-uncased', max_length=32)
example = encoder.encode('The battery life is amazing but the screen is dim', 'battery life', label=2)

{k: tuple(v.shape) for k, v in example.items()}

In [ ]:
tokens = encoder.tokenizer.convert_ids_to_tokens(example['input_ids'])

pd.DataFrame({
    'token': tokens,
    'type': example['token_type_ids'].tolist(),
    'attn': example['attention_mask'].tolist(),
    'context': example['context_mask'].tolist(),
    'aspect': example['aspect_mask'].tolist(),
}).head(18)

In [ ]:
# The two masks never overlap and never select a special or padding token.
ctx, asp = example['context_mask'], example['aspect_mask']
print('overlap          :', bool((ctx & asp).any()))
print('context tokens   :', encoder.tokenizer.convert_ids_to_tokens(example['input_ids'][ctx]))
print('aspect tokens    :', encoder.tokenizer.convert_ids_to_tokens(example['input_ids'][asp]))

## Dataset and class weights

In [ ]:
from src.data.dataset import ABSADataset, collate_batch

dataset = ABSADataset(val_df.head(64), encoder, label_map)
batch = collate_batch([dataset[i] for i in range(8)])

print(dataset)
print({k: tuple(v.shape) for k, v in batch.items()})
print('class weights:', dict(zip(label_map.labels, dataset.class_weights().tolist())))

## Next

```bash
cd ..
python train.py --config configs/bert_base.yaml
```

Then load the run and inspect its errors:

```python
from src.artifact import load_artifact
from src.evaluation.evaluate import error_analysis

run = load_artifact('../artifacts/bert-base-aspect-context')
```